# Rozhranie (interface)

**Rozhranie** definuje kontrakt obsahujúci určité funkcionality, ktoré musí neabstraktná trieda alebo štruktúra implementovať. T. j. musí poskytnúť implementáciu členov z daného rozhrania, ktoré implementuje. Rozhranie je ako abstraktná základná trieda s abstraktnými členmi.

- rozhranie môže byť deklarované v mennom priestori, v triede alebo v štruktúre
- konvencia je písať ho s prvým veľkým písmenom I
- rozhranie môže obsahovať verejné členy (zvyčajne bez teľa):
    - metódy
    - vlastnosti
    - indexery
    - udalosti
- rozhranie môže byť dediť od jedného rozhrania
- trieda alebo štruktúra môže implementovať jedno alebo aj viacero rozhraní naraz
- od C# 8.0 môže interface obsahovať vo svojom vnútri aj implementáciu niektorých členov alebo aj všetky členy

Príklad vlastného definovaného rozhrania:

In [ ]:
interface ISampleInterface
{
    void SampleMethod();
}

class ImplementationClass : ISampleInterface
{
    void ISampleInterface.SampleMethod()
    {
        Console.WriteLine("SampleMethod :)");
    }
}

var obj = new ImplementationClass();
((ISampleInterface)obj).SampleMethod();


Príklad generického rozhrania [`System.IEquatable<T>`](https://docs.microsoft.com/en-us/dotnet/api/system.iequatable-1) (implicitná implementácia):

In [ ]:
interface IEquatable<T>
{
    bool Equals(T obj);
}

class Person : IEquatable<Person>, IEquatable<int>
{
    public string FirstName {get; set;}
    public string LastName { get; set; }
    public DateTime Birthdate { get; set; }    

    public bool Equals(Person person)
    {
       return (FirstName, LastName, Birthdate) == (person.FirstName, person.LastName, person.Birthdate);
    }

    public bool Equals(int year)
    {
        return year.Equals(Birthdate.Year);
    }
}

var p1 = new Person { FirstName = "Jan", LastName = "Mrkvicka", Birthdate = DateTime.Today };
var p2 = new Person { FirstName = "Jan", LastName = "Mrkvicka", Birthdate = DateTime.Today };

Console.WriteLine(p1.Equals(p2));
Console.WriteLine(p1.Equals(DateTime.Today.Year));

## Explicitná implementácia rozhrania

Explicitná implementácia používa v názve implementovanej metódy alebo vlastnosti identifikátor rozhrania (v nasledujúcom príklade `IEquatable<Person>`), pričom nie je povolený prístupový modifikátor (`bool IEquatable<Person>.Equals(Person person)`):  

In [ ]:
class Person : IEquatable<Person>
{
    public string FirstName {get; set;}
    public string LastName { get; set; }
    public DateTime Birthdate { get; set; }    

    bool IEquatable<Person>.Equals(Person person)
    {
        return (FirstName, LastName, Birthdate) == (person.FirstName, person.LastName, person.Birthdate);
    }
}

var p1 = new Person { FirstName = "Jan", LastName = "Mrkvicka", Birthdate = DateTime.Today };
var p2 = new Person { FirstName = "Jan", LastName = "Mrkvicka", Birthdate = DateTime.Today };

Console.WriteLine(p1.Equals(p2)); // Tu sa volá metóda Equals(object) z predka object (System.Object)

var p1AsIEquatable = (IEquatable<Person>)p1;
Console.WriteLine(p1AsIEquatable.Equals(p2)); // Tu sa volá implementovaná metóda z rozhrania IEquatable<Person>



Iný príklad:

In [ ]:
class MyData : IEnumerable<int>
{
    private List<int> _numbers = new List<int>();

    public IEnumerator<int> GetEnumerator()
    {
        return _numbers.GetEnumerator();
    }

    IEnumerator IEnumerable.GetEnumerator()
    {
        return GetEnumerator();
    }
}

## Rozhrania z .NETu

### Rozhranie `IDisposable`

Poskytuje priestor na upratanie a uvolnenie vzácnych zdrojov, najmä nemanažovaných, ktoré .NET nevie upratať. Používa sa pre uvoľnenie pamäte a iných vzácnych zdrojov, keď už s nimi nepotrebujeme pracovať (napr. pri práci s databázou, súbormi, ...).

```csharp
public interface IDisposable
{
    void Dispose();
}
```

Ak nejaká trieda implementujte toto rozhranie, mali by sme volať metódu `Dispose()` manuálne (prípadne `Close`, ak existuje - väčšinou sa ale volá v Dispose()) alebo ešte lepšie s využitím príkazu `using`, ktorý `Dispose` metódu zavolá automaticky (pozor: nie s direktívou `using` na zavedenie menných priestorov):

In [ ]:
using System.IO;

void WriteTextFile(string path)
{
    // Starší typ zápisu príkazu using - platí v novom vytvorenom bloku
    using (StreamWriter sw = File.CreateText(path))
    {
        sw.WriteLine("Ahoj");
        sw.WriteLine("zo");
        sw.WriteLine("suboru");        
    } // Tu sa zavola sw.Dispose() automaticky
}

void ReadTextFile(string path)
{
    // Novší typ príkazu using na riadok bez zložených zátvoriek 
    using var sr = File.OpenText(path);
    string s;
    while ((s = sr.ReadLine()) != null)
    {
        Console.WriteLine(s);
    }    
} // Tu sa zavola sr.Dispose() automaticky

string fileName = "TextFile.txt";
WriteTextFile(fileName);
ReadTextFile(fileName);

Príklad pre uvoľnenie zdrojov s použitím `Dispose()` metódy a odporúčaneho vzoru (Dispose pattern), ktorý pridáva finalizér (deštruktor) a virtuálnu protected metódu `Dispose(bool)`:

In [ ]:
using System;
using System.ComponentModel;

public class MyResource : IDisposable
{
    // Smerník na externý nemanažovaný zdroj, ktorý po skončení používania treba uvoľniť
    private IntPtr handle;
    // Príklad na iný manažovaný zdroj, ktorý je vhodné uvoľniť skôr 
    private Component component = new Component();
    private bool disposed = false;

    public MyResource(IntPtr handle)
    {
        this.handle = handle;
    }

    public void Dispose()
    {
        Dispose(true);
        // Zabraňuje, aby sa duplicitne nevolal Garbage Collector
        GC.SuppressFinalize(this); 
    }

    protected virtual void Dispose(bool disposing)
    {
        if (!this.disposed)
        {
            if (disposing)
            {
                component.Dispose();
            }

            CloseHandle(handle);
            handle = IntPtr.Zero;

            disposed = true;
        }
    }

    [System.Runtime.InteropServices.DllImport("Kernel32")]
    private extern static Boolean CloseHandle(IntPtr handle);

    // C# finalizér (deštruktor) - ak sa nezavolala metóda Dispose, Garbage Collector tu má šancu zdroje upratať 
    // Je to ale v nedefinovanom čase a môže to byť už neskoro
    ~MyResource()
    {        
        Dispose(false);
    }
}

### Rozhrania `IComparable` a `IComparable<T>`

In [ ]:
public class Temperature : IComparable<Temperature>
{
    public int CompareTo(Temperature obj) 
    {
        if (obj == null) 
            return 1;

        return this.Fahrenheit.CompareTo(obj.Fahrenheit);
    }

    public double Fahrenheit { get; set; }

    public double Celsius
    {
        get =>  (Fahrenheit - 32) * (5.0/9);
        set => Fahrenheit = (value * 9.0/5) + 32;
    }
}

### Rozhrania `IEnumerable` a `IEnumerable<T>`

Umožňujú prechádzať a získavať hodnoty v kolekcii sekvenčne. Vďaka nim je možné využiť aj cyklus foreach na prechádzanie vlastných tried.  
Definícia rozhraní `IEnumerable` a `IEnumerable<T>` v knižnici .NETu: 

```csharp
public interface IEnumerable
{
    IEnumerator GetEnumerator();
}

public interface IEnumerable<out T> : IEnumerable
{
    IEnumerator<T> GetEnumerator();
}
```

Ak si rozhranie `IEnumerable` alebo `IEnumerable<T>` implementujete, umožníte prechádzať dáta nielen pomocou enumerátora, ale aj pomocou cyklu foreach. Ten automaticky volá nad kolekciou metódu `GetEnumerator()`, ktorá vracia inštanciu enumerátora implementujúceho rozhranie `IEnumerator` / `IEnumerator<T>`:

```csharp
public interface IEnumerator
{
    bool MoveNext();
    object Current { get; }
    void Reset();
}

public interface IEnumerator<out T> : IDisposable, IEnumerator
{
    T Current { get; }
}
```

 Následne volá metódu `MoveNext()`, ktorou sa posunie vpred na prvý záznam a ak existuje, vráti hodnotu `True`. Ak žiaden záznam neexistuje alebo sme prišli už na koniec, metóda vráti hodnotu `False`. V prípade, že vrátil `True`, môže získať hodnotu pomocou vlastnosti `Current`.  ktoré foreach cyklicky volá pre získanie hodnôt položiek dovtedy, kým metóda `MoveNext()` nevráti hodnotu `False`



In [ ]:
// Vygeneruje 6 hodnôt od nuly { 0, 1, 2, 3, 4, 5 };
//IEnumerable<int> numbers = Enumerable.Range(0, 6); 

// Alebo môžeme použiť:
//int[] numbers = { 0, 1, 2, 3, 4, 5 };
List<int> numbers = new List<int>() { 0, 1, 2, 3, 4, 5 };

Console.WriteLine("Výpis čísiel automaticky cez foreach:");
foreach (var number in numbers)
{
    Console.WriteLine(number);
}

Console.WriteLine("Výpis čísiel manuálnym volaním enumerátora:");
IEnumerator<int> enumerator = numbers.GetEnumerator();
while (enumerator.MoveNext())
{
    Console.WriteLine(enumerator.Current);
}

#### Kľúčové slovo yield

`yield` – indikuje, že metóda, operátor alebo get časť, v ktorých sa môže nachádzať, je iterátorom
- ich návratový typ musí byť: `IEnumerable`, `IEnumerable<T>`, `IEnumerator`, alebo `IEnumerator<T>`
- v ich deklarácii nesmie byť použitý parameter s kľúčovým slovom `in`, `ref` alebo `out` 
- nedá sa použiť v anonymných metódach a metódach označených ako `unsafe`

Zavedené od C# 2, odpadá nutnosť manuálneho vytvárania enumerátora (vlastnej triedy implementujúcej rozhranie `IEnumerator`).

Použitie:
- `yield return` výraz; - vráti jeden prvok v jednom čase
- `yield break`; - ukončí iterovanie 


In [ ]:
public IEnumerable<int> GetNumbers()
{
    yield return 0;
    yield return 1;
    yield return 2;
    yield return 3;
    yield return 4;
    yield return 5;
}

Console.WriteLine("foreach:");
foreach (var number in GetNumbers())
{
    Console.WriteLine(number);
}

Console.WriteLine("while:");
IEnumerator<int> enumerator = GetNumbers().GetEnumerator();
while (enumerator.MoveNext())
{
    Console.WriteLine(enumerator.Current);
}

In [ ]:
public IEnumerable<int> GetReverseNumbers(int count)
{
    for (int i = 0; i < count; i++)
        yield return i;    
}

Console.WriteLine("foreach:");
foreach (var number in GetReverseNumbers(6))
{
    Console.WriteLine(number);
}

Console.WriteLine("while:");
IEnumerator<int> enumerator = GetReverseNumbers(6).GetEnumerator();
while (enumerator.MoveNext())
{
    Console.WriteLine(enumerator.Current);
}

Iný príklad:

In [ ]:
public IEnumerable<int> Power(int number, int exponent)
{
    int result = 1;

    for (int i = 0; i < exponent; i++)
    {
        result = result * number;
        yield return result;
    }
}

// Zobrazí postupne mocniny 2^𝑛 pre ∀𝑛∈⟨1,8⟩
// Vypíše: 2 4 8 16 32 64 128 256
foreach (int i in Power(2, 8))
{
    Console.Write("{0} ", i);
}

Komplexnejší príklad:

In [ ]:
class MyData : IEnumerable<int>
{
    private List<int> _numbers = new List<int>();

    public void Add(int number) => _numbers.Add(number);

    // Rozhranie IEnumerable<int> implementované implicitne:
    public IEnumerator<int> GetEnumerator()
    {
        return _numbers.GetEnumerator();
        // Alebo môžeme vrátiť aj enumerátor z metódy vracajúcej IEnumerable<int>
        //return GetReverse().GetEnumerator();
    }

    // Rozhranie IEnumerable implementované explicitne:
    IEnumerator IEnumerable.GetEnumerator()
    {
        return GetEnumerator();
    }

    public IEnumerable<int> GetReverse()
    {
        for (int i = _numbers.Count - 1; i >= 0; i--)
        {
            yield return _numbers[i];
        }
    }

    public IEnumerable<int> GetDoubled()
    {
        foreach (var number in _numbers)
        {
            yield return number * 2;
        }
    }

    public IEnumerable<int> GetEverySecondNumbers()
    {
        bool use = false;
        foreach (var number in _numbers)
        {
            use = !use;
            if (use)
                continue;

            yield return number;
        }
    }

    public IEnumerable<int> GetOnlyPositiveNumbers()
    {
        foreach (var number in _numbers)
        {
            if (number > 0)
                yield return number;
        }
    }

    public IEnumerable<int> GetFirstNumbers(int maxNumberCount)
    {
        int count = 0;
        foreach (var number in _numbers)
        {
            yield return number;

            count++;
            if (count == maxNumberCount)
                yield break; // Ukončí enumerovanie
        }
    }
}


In [ ]:
var data = new MyData();
data.Add(1);
data.Add(-2);
data.Add(3);
data.Add(4);
data.Add(-5);

string output = string.Empty;
foreach (var d in data) 
{
    output += d + " ";
}
Console.WriteLine($"Foreach: {output}");

output = string.Empty;
foreach (var d in data.GetOnlyPositiveNumbers()) 
{
    output += d + " ";
}
Console.WriteLine($"Len kladné čísla: {output}");

In [ ]:
public void Display<T>(string text, IEnumerable<T> enumerable)
{
    string output = string.Empty;
    foreach (var d in enumerable) 
    {
        output += d + " ";
    }

    Console.WriteLine($"{text}: {output}");
}

Display("Foreach", data);
//Display("Foreach", data.GetEnumerator()); // Error CS150
Display("Opačné poradie", data.GetReverse());
Display("Vynásobené číslom 2", data.GetDoubled());
Display("Každé druhé číslo", data.GetEverySecondNumbers());
Display("Len kladné čísla", data.GetOnlyPositiveNumbers());
Display("Prvé tri čísla", data.GetFirstNumbers(3));

## Odkazy
Microsoft dokumentácia:
- [Rozhrania](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/types/interfaces)
- [Kľúčové slovo interface](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/keywords/interface)